# Data Cleaning & Transformation

## 1. Objective

The purpose of this stage is to prepare the Maven Market dataset for analysis by identifying and handling data quality issues, correcting data types, and ensuring consistency across tables.

## 2. Data Preparation

* Load the original CSV files.
* Create working copies of the datasets.
* Keep the raw data unchanged to preserve the original source files.

## 3. Cleaning Tasks

The cleaning process will focus on:

* **Missing Values:** Investigate and handle missing values where appropriate.
* **Duplicate Records:** Review potential duplicate records before deciding whether to remove them.
* **Data Types:** Convert date columns from strings to datetime.
* **Data Consistency:** Standardize inconsistent values and formats where necessary.
* **Data Validation:** Check identifiers, numerical values, and relationships between tables.

## 4. Expected Outcome

Produce clean and consistent datasets ready for the next stages:

* Database Storage (SQL Server).
* Data Modeling.
* Business Analysis.
* Power BI Dashboard Development.

## 5. Important Notes

* Original raw files will remain unchanged.
* No records will be deleted without investigation.
* Missing values will not be replaced without understanding their meaning.
* Every transformation will be validated before saving the processed data.


In [6]:
import pandas as pd
import numpy as np
import os

## Read Row Data

In [7]:
raw_path = "../Data/Raw Data"
Raw = os.listdir(raw_path)

Raw


['MavenMarket_Calendar.csv',
 'MavenMarket_Customers.csv',
 'MavenMarket_Products.csv',
 'MavenMarket_Regions.csv',
 'MavenMarket_Returns_1997-1998.csv',
 'MavenMarket_Stores.csv',
 'MavenMarket_Transactions_1997.csv',
 'MavenMarket_Transactions_1998.csv']

## 1. Loading Data

In this step, we load the original CSV files into Pandas DataFrames to prepare them for the cleaning process.

### Loading Function
A reusable function called `load_data()` is created to simplify the loading process.

- `os.path.join()`: Combines the folder path and file name to create the full file path.
- `pd.read_csv()`: Reads the CSV file and converts it into a Pandas DataFrame.
- `return`: Returns the loaded DataFrame.

### Loading Datasets
The function is used to load the following datasets:
- Calendar
- Customers
- Products
- Regions
- Returns
- Stores
- Transactions (1997 and 1998)

### Combining Transactions
The two Transactions datasets are combined vertically using `pd.concat()`.

`ignore_index=True` resets the index to create a continuous sequence after combining the two datasets.

### Important Note
The original raw data is kept unchanged. All cleaning operations will be performed on working copies.

In [8]:
def load_data(path,file_name):
    file_path = os.path.join(raw_path, file_name)
    return pd.read_csv(file_path)

In [9]:
calendar = load_data(raw_path, "MavenMarket_Calendar.csv")

customers = load_data(raw_path, "MavenMarket_Customers.csv")

products = load_data(raw_path, "MavenMarket_Products.csv")

regions = load_data(raw_path, "MavenMarket_Regions.csv")

returns = load_data(raw_path, "MavenMarket_Returns_1997-1998.csv")

stores = load_data(raw_path, "MavenMarket_Stores.csv")

transactions_1997 = load_data(raw_path, "MavenMarket_Transactions_1997.csv")

transactions_1998 = load_data(raw_path, "MavenMarket_Transactions_1998.csv")

### Combining Transactions

The Transactions dataset is divided into two separate CSV files (1997 and 1998).

Since both files have the same structure and columns, they are combined vertically into a single DataFrame using `pd.concat()`.

- `pd.concat()`: Combines multiple DataFrames.
- `ignore_index=True`: Resets the index after concatenation.

The resulting DataFrame contains transactions from both years.

In [10]:
transactions = pd.concat([transactions_1997, transactions_1998], ignore_index=True)

### Creating Working Copies

Before starting the cleaning process, we create independent copies of the original DataFrames.

This allows us to perform cleaning and transformations without modifying the originally loaded data.

The original DataFrames will be preserved for comparison and validation.

In [11]:
calendar_clean = calendar.copy()
customers_clean = customers.copy()
products_clean = products.copy()
regions_clean = regions.copy()
returns_clean = returns.copy()
stores_clean = stores.copy()
transactions_clean = transactions.copy()


### Data Loading Validation

After loading and combining the datasets, we validate their dimensions to ensure that all files were loaded correctly.

The `shape` attribute returns:
- Number of rows.
- Number of columns.

This validation helps confirm that the loaded datasets match the expected structure identified during the Data Exploration phase.

In [12]:
tables = {
    "Calendar": calendar_clean,
    "Customers": customers_clean,
    "Products": products_clean,
    "Regions": regions_clean,
    "Returns": returns_clean,
    "Stores": stores_clean,
    "Transactions": transactions_clean
}



for name,df in tables.items():
    print(f"Table: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Missing Values:\n{df.isnull().sum()}")
    print("-"*50)   
    

Table: Calendar
Shape: (730, 1)
Columns: ['date']
Missing Values:
date    0
dtype: int64
--------------------------------------------------
Table: Customers
Shape: (10281, 20)
Columns: ['customer_id', 'customer_acct_num', 'first_name', 'last_name', 'customer_address', 'customer_city', 'customer_state_province', 'customer_postal_code', 'customer_country', 'birthdate', 'marital_status', 'yearly_income', 'gender', 'total_children', 'num_children_at_home', 'education', 'acct_open_date', 'member_card', 'occupation', 'homeowner']
Missing Values:
customer_id                0
customer_acct_num          0
first_name                 0
last_name                  1
customer_address           0
customer_city              0
customer_state_province    0
customer_postal_code       0
customer_country           0
birthdate                  0
marital_status             0
yearly_income              0
gender                     0
total_children             0
num_children_at_home       0
education          

### Data Loading Validation – Summary

All seven datasets were successfully loaded and validated.

- The Transactions datasets (1997 and 1998) were successfully combined into a single DataFrame containing 269,720 rows and 6 columns.
- All dataset dimensions match the results obtained during the Data Exploration phase.
- Missing values were identified in:
  - `Customers.last_name`: 1 missing value.
  - `Products.recyclable`: 687 missing values.
  - `Products.low_fat`: 1,008 missing values.
- No missing values were found in the remaining datasets.

The datasets are ready for the cleaning and transformation process.

In [13]:
for name,df in tables.items():
    print(f"Table: {name}")
    print(f"Data Types:\n {df.dtypes}")
    print(""*50)

Table: Calendar
Data Types:
 date    str
dtype: object

Table: Customers
Data Types:
 customer_id                int64
customer_acct_num          int64
first_name                   str
last_name                    str
customer_address             str
customer_city                str
customer_state_province      str
customer_postal_code       int64
customer_country             str
birthdate                    str
marital_status               str
yearly_income                str
gender                       str
total_children             int64
num_children_at_home       int64
education                    str
acct_open_date               str
member_card                  str
occupation                   str
homeowner                    str
dtype: object

Table: Products
Data Types:
 product_id                int64
product_brand               str
product_name                str
product_sku               int64
product_retail_price    float64
product_cost            float64
product_weight    

In [14]:
date_columns = {
    "Calendar": ["date"],
    "Customers": ["birthdate", "acct_open_date"],
    "Returns": ["return_date"],
    "Stores": ["first_opened_date", "last_remodel_date"],
    "Transactions": ["transaction_date", "stock_date"]
}

for table_name, columns in date_columns.items():
    for col in columns:
        tables[table_name][col] = pd.to_datetime(
            tables[table_name][col],
            errors="coerce"
        )

In [15]:
customers_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10281 entries, 0 to 10280
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customer_id              10281 non-null  int64         
 1   customer_acct_num        10281 non-null  int64         
 2   first_name               10281 non-null  str           
 3   last_name                10280 non-null  str           
 4   customer_address         10281 non-null  str           
 5   customer_city            10281 non-null  str           
 6   customer_state_province  10281 non-null  str           
 7   customer_postal_code     10281 non-null  int64         
 8   customer_country         10281 non-null  str           
 9   birthdate                10281 non-null  datetime64[us]
 10  marital_status           10281 non-null  str           
 11  yearly_income            10281 non-null  str           
 12  gender                   10281 non-null  st

In [16]:
customers_clean.isna().sum()

customer_id                0
customer_acct_num          0
first_name                 0
last_name                  1
customer_address           0
customer_city              0
customer_state_province    0
customer_postal_code       0
customer_country           0
birthdate                  0
marital_status             0
yearly_income              0
gender                     0
total_children             0
num_children_at_home       0
education                  0
acct_open_date             0
member_card                0
occupation                 0
homeowner                  0
dtype: int64

In [17]:
customers_clean = customers_clean.dropna()

In [18]:
customers_clean.isna().sum()

customer_id                0
customer_acct_num          0
first_name                 0
last_name                  0
customer_address           0
customer_city              0
customer_state_province    0
customer_postal_code       0
customer_country           0
birthdate                  0
marital_status             0
yearly_income              0
gender                     0
total_children             0
num_children_at_home       0
education                  0
acct_open_date             0
member_card                0
occupation                 0
homeowner                  0
dtype: int64

In [19]:
customers_clean.shape

(10280, 20)

In [20]:
products_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_id            1560 non-null   int64  
 1   product_brand         1560 non-null   str    
 2   product_name          1560 non-null   str    
 3   product_sku           1560 non-null   int64  
 4   product_retail_price  1560 non-null   float64
 5   product_cost          1560 non-null   float64
 6   product_weight        1560 non-null   float64
 7   recyclable            873 non-null    float64
 8   low_fat               552 non-null    float64
dtypes: float64(5), int64(2), str(2)
memory usage: 109.8 KB


In [21]:
products_clean.isna().sum()

product_id                 0
product_brand              0
product_name               0
product_sku                0
product_retail_price       0
product_cost               0
product_weight             0
recyclable               687
low_fat                 1008
dtype: int64

In [22]:
products_clean[["recyclable","low_fat"]]

,recyclable,low_fat
0,NaN,NaN
1,NaN,1.0
2,1.0,1.0
3,1.0,NaN
4,1.0,NaN
...,...,...
1555,1.0,1.0
1556,1.0,NaN
1557,NaN,1.0
1558,NaN,NaN


`Missing = 0 means NOT Recyclable And Low_fat`

==> Convert Missing to " 0 "

In [23]:
products_clean = products_clean.fillna(0)

In [24]:
products_clean[["recyclable","low_fat"]]

,recyclable,low_fat
0,0.0,0.0
1,0.0,1.0
2,1.0,1.0
3,1.0,0.0
4,1.0,0.0
...,...,...
1555,1.0,1.0
1556,1.0,0.0
1557,0.0,1.0
1558,0.0,0.0


`Convert Columns Type To int`

In [25]:
products_clean[["recyclable","low_fat"]] = products_clean[["recyclable","low_fat"]].astype(int)

In [26]:
products_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_id            1560 non-null   int64  
 1   product_brand         1560 non-null   str    
 2   product_name          1560 non-null   str    
 3   product_sku           1560 non-null   int64  
 4   product_retail_price  1560 non-null   float64
 5   product_cost          1560 non-null   float64
 6   product_weight        1560 non-null   float64
 7   recyclable            1560 non-null   int64  
 8   low_fat               1560 non-null   int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 109.8 KB


In [27]:
regions_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   region_id       109 non-null    int64
 1   sales_district  109 non-null    str  
 2   sales_region    109 non-null    str  
dtypes: int64(1), str(2)
memory usage: 2.7 KB


In [28]:
regions_clean.head()

,region_id,sales_district,sales_region
0,1,San Francisco,Central West
1,2,Mexico City,Mexico Central
2,3,Los Angeles,South West
3,4,Guadalajara,Mexico West
4,5,Vancouver,Canada West


In [29]:
returns_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7087 entries, 0 to 7086
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   return_date  7087 non-null   datetime64[us]
 1   product_id   7087 non-null   int64         
 2   store_id     7087 non-null   int64         
 3   quantity     7087 non-null   int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 221.6 KB


In [30]:
returns_clean.head()

,return_date,product_id,store_id,quantity
0,1997-01-01,250,6,1
1,1997-01-01,628,6,1
2,1997-01-01,869,6,1
3,1997-01-02,469,11,1
4,1997-01-02,532,23,2


In [31]:
stores_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   store_id              24 non-null     int64         
 1   region_id             24 non-null     int64         
 2   store_type            24 non-null     str           
 3   store_name            24 non-null     str           
 4   store_street_address  24 non-null     str           
 5   store_city            24 non-null     str           
 6   store_state           24 non-null     str           
 7   store_country         24 non-null     str           
 8   store_phone           24 non-null     str           
 9   first_opened_date     24 non-null     datetime64[us]
 10  last_remodel_date     24 non-null     datetime64[us]
 11  total_sqft            24 non-null     int64         
 12  grocery_sqft          24 non-null     int64         
dtypes: datetime64[us](2), int64(4), s

In [32]:
stores_clean.head()

,store_id,region_id,store_type,store_name,store_street_address,store_city,store_state,store_country,store_phone,first_opened_date,last_remodel_date,total_sqft,grocery_sqft
0,1,28,Supermarket,Store 1,2853 Bailey Rd,Acapulco,Guerrero,Mexico,262-555-5124,1982-01-09,1990-12-05,23593,17475
1,2,78,Small Grocery,Store 2,5203 Catanzaro Way,Bellingham,WA,USA,605-555-8203,1970-04-02,1973-06-04,28206,22271
2,3,76,Supermarket,Store 3,1501 Ramsey Circle,Bremerton,WA,USA,509-555-1596,1959-06-14,1967-11-19,39696,24390
3,4,27,Gourmet Supermarket,Store 4,433 St George Dr,Camacho,Zacatecas,Mexico,304-555-1474,1994-09-27,1995-12-01,23759,16844
4,5,4,Small Grocery,Store 5,1250 Coggins Drive,Guadalajara,Jalisco,Mexico,801-555-4324,1978-09-18,1991-06-29,24597,15012


In [33]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 269720 entries, 0 to 269719
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   transaction_date  269720 non-null  str  
 1   stock_date        269720 non-null  str  
 2   product_id        269720 non-null  int64
 3   customer_id       269720 non-null  int64
 4   store_id          269720 non-null  int64
 5   quantity          269720 non-null  int64
dtypes: int64(4), str(2)
memory usage: 12.3 MB


In [34]:
transactions[["transaction_date", "stock_date"]] = pd.to_datetime(["transaction_date", "stock_date"], errors="coerce")

C:\Users\ip\AppData\Local\Temp\ipykernel_9980\730989349.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  transactions[["transaction_date", "stock_date"]] = pd.to_datetime(["transaction_date", "stock_date"], errors="coerce")


`Load Cleand Data`

In [ ]:
calendar_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Calendar.csv", index = False)
customers_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Customers.csv", index = False)
products_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Products.csv", index = False)
regions_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Regions.csv", index = False)
returns_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Returns.csv", index = False)
stores_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Stores.csv", index = False)
transactions_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Transactions(1997 - 1998).csv", index = False)

In [37]:
returns_clean.to_csv(r"D:\Data Analysis\02 - Projects\Maven_Market_End_to_End\Data\Processed\Returns.csv", index = False)
